In [104]:
import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('.'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

./.DS_Store
./notebook_01.ipynb
./notebook_02.ipynb


In [129]:
# Train DataSet
train_data = pd.read_csv("../data/train.csv")
train_data

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [106]:
# Test DataSet
test_data = pd.read_csv("../data/test.csv")
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [ ]:
# Is there any NaN?
train_data.isna().any()
#test_data.isna().any()

PassengerId    False
Survived       False
Pclass         False
Name           False
Sex            False
Age             True
SibSp          False
Parch          False
Ticket         False
Fare           False
Cabin           True
Embarked        True
dtype: bool

In [108]:
# Fill Fare
test_data["Fare"] = test_data["Fare"].fillna(test_data["Fare"].median())
test_data.isna().any()

PassengerId    False
Pclass         False
Name           False
Sex            False
Age             True
SibSp          False
Parch          False
Ticket         False
Fare           False
Cabin           True
Embarked       False
dtype: bool

In [109]:
# Fill Ages
middle_age = train_data["Age"].mean()

print(middle_age)

train_data["Age"] = train_data['Age'].fillna(middle_age)
test_data["Age"] = test_data['Age'].fillna(middle_age)

train_data.isna().any()
#test_data.isna().any()

29.69911764705882


PassengerId    False
Survived       False
Pclass         False
Name           False
Sex            False
Age            False
SibSp          False
Parch          False
Ticket         False
Fare           False
Cabin           True
Embarked        True
dtype: bool

In [110]:
# Sex feature
print(train_data.groupby('Sex')['Survived'].mean())

Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64


In [111]:
# Priority function
def is_priority(passanger):
    age, sex = passanger
    if age < 16 or sex == "female":
        return 1
    else:
        return 0

In [112]:
# Add priority
train_data["IsPriority"] = train_data[["Age", "Sex"]].apply(is_priority, axis=1)
test_data["IsPriority"] = test_data[["Age", "Sex"]].apply(is_priority, axis=1)

In [113]:
# Features
features = ["Pclass", "Sex", "SibSp", "Parch", "Age", "IsPriority", "Fare"]
print(f"Features: {features}")

Features: ['Pclass', 'Sex', 'SibSp', 'Parch', 'Age', 'IsPriority', 'Fare']


In [114]:
# X, y
X = pd.get_dummies(train_data[features])
y = train_data["Survived"]

kaggle_X = pd.get_dummies(test_data[features])

X

,Pclass,SibSp,Parch,Age,IsPriority,Fare,Sex_female,Sex_male
0,3,1,0,22.000000,0,7.2500,False,True
1,1,1,0,38.000000,1,71.2833,True,False
2,3,0,0,26.000000,1,7.9250,True,False
3,1,1,0,35.000000,1,53.1000,True,False
4,3,0,0,35.000000,0,8.0500,False,True
...,...,...,...,...,...,...,...,...
886,2,0,0,27.000000,0,13.0000,False,True
887,1,0,0,19.000000,1,30.0000,True,False
888,3,1,2,29.699118,1,23.4500,True,False
889,1,0,0,26.000000,0,30.0000,False,True


In [115]:
# Test
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=99)

model = RandomForestClassifier(n_estimators=500, max_depth=6, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
score = accuracy_score(y_test, predictions)

print(f"Accuracy score {score:.6f}")

Accuracy score 0.798507


In [ ]:
# Final
final_model = RandomForestClassifier(n_estimators=500, max_depth=5, random_state=42)
final_model.fit(X, y)

final_predictions = final_model.predict(kaggle_X)

In [126]:
# Cross validation
from sklearn.model_selection import cross_val_score
scores = cross_val_score(final_model, X, y, cv=5)
print(f"Median accuracy score: {scores.mean():.4f}")

Median accuracy score: 0.8283


In [128]:
# Output
output = pd.DataFrame({
    'PassengerId': test_data.PassengerId,
    'Survived': final_predictions
    })
output.to_csv('../submissions/submission_08.csv', index=False)
print("Submission succesfully saved")

Submission succesfully saved
